![Bronze-to-silver.jpg](./Bronze-to-silver.jpg "Bronze-to-silver.jpg")

In [0]:
# 1. Detectar usuario
current_user = spark.sql("SELECT current_user()").first()[0]
base_path = "/Volumes/ecomotive/landing/ecomotive_vol"

# 2. Rutas de Checkpoints para Silver (Sistema)
# Nota: Silver se guarda en tablas gestionadas (Managed Tables), 
# por lo que no definimos ruta de "data", solo de "checkpoints".
chk_silver_drivers = f"{base_path}/sys/silver_drivers/checkpoints"
chk_silver_sensors = f"{base_path}/sys/silver_sensors/checkpoints"

# 3. Limpieza de Checkpoints (Para empezar la práctica de cero)
dbutils.fs.rm(chk_silver_drivers, True)
dbutils.fs.rm(chk_silver_sensors, True)
spark.sql("DROP TABLE IF EXISTS ecomotive.silver.silver_drivers")
spark.sql("DROP TABLE IF EXISTS ecomotive.silver.silver_sensors")

print(f"Rutas de sistema configuradas para usuario: {current_user}")

Rutas de sistema configuradas para usuario: pablo.cumbreraconde@gmail.com


In [0]:
from pyspark.sql.functions import col, trim, initcap, regexp_replace, to_date, split, current_timestamp

# 1. READ STREAM
df_bronze_drivers = spark.readStream.table("ecomotive.bronze.bronze_drivers")

# 2. TRANSFORMACIONES
df_silver_drivers = (df_bronze_drivers
  # A) Limpieza de Nombres
  .withColumn("driver_name", initcap(trim(col("raw_name"))))
  
  # B) Limpieza de Licencia (NUEVO)
  # Usamos trim para evitar " LIC-123 "
  .withColumn("license_ref", trim(col("license_ref"))) 
  
  # C) Limpieza de Salario
  .withColumn("hourly_wage", 
              regexp_replace(col("hourly_wage_text"), " EUR", "").cast("double"))
  
  # D) Fecha y Certificaciones
  .withColumn("hire_date", to_date(col("hired_date"), "yyyy-MM-dd"))
  .withColumn("certifications", split(col("certifications_list"), "-"))
  
  # E) Auditoría
  .withColumn("processed_at", current_timestamp())
  
  # F) Selección Final
  .select(
      col("driver_id"),
      col("driver_name"),
      col("license_ref"),   
      col("hourly_wage"),
      col("certifications"),
      col("hire_date"),
      col("processed_at")
  )
)

# 3. WRITE STREAM
query_drivers = (df_silver_drivers.writeStream
  .format("delta")
  .option("checkpointLocation", chk_silver_drivers)
  .outputMode("append")
  .trigger(availableNow=True)
  .table("ecomotive.silver.silver_drivers")
)

query_drivers.awaitTermination()
print("Silver Drivers actualizado: Incluye Licencia limpia.")

Silver Drivers actualizado: Incluye Licencia limpia.


In [0]:
from pyspark.sql.functions import col, to_timestamp

# 1. READ STREAM
df_bronze_sensors = spark.readStream.table("ecomotive.bronze.bronze_sensors")

# 2. TRANSFORMACIONES
df_silver_sensors = (df_bronze_sensors
  # A) Aplanar Estructuras (Dot Notation)
  # Sacamos los hijos del struct "engine_data" a columnas de primer nivel
  .withColumn("rpm", col("engine_data.rpm"))
  .withColumn("temperature", col("engine_data.temperature_c"))
  .withColumn("engine_status", col("engine_data.status"))
  
  # B) Castear Tiempo
  .withColumn("event_time", to_timestamp(col("timestamp")))
  
  # C) Seleccionar columnas (incluyendo las nuevas del Batch 02 si existen)
  # Nota: En producción, seríamos más explícitos, pero aquí seleccionamos lo útil.
  .select(
      "event_id",
      "sensor_id",
      "driver_id",
      "event_time",
      "rpm",
      "temperature",
      "engine_status",
      "battery_level",
      col("location.lat").alias("latitude"),  # Aplanamos estructura
      col("location.lon").alias("longitude"),  # Aplanamos estructura
      "error_codes" # Dejamos el array de errores tal cual
  )
)

# 3. WRITE STREAM
query_sensors = (df_silver_sensors.writeStream
  .format("delta")
  .option("checkpointLocation", chk_silver_sensors)
  .option("mergeSchema", "true") # Por si las moscas con nuevas columnas
  .trigger(availableNow=True)
  .table("ecomotive.silver.silver_sensors")
)

query_sensors.awaitTermination()
print("Silver Sensores: Estructuras Aplanadas.")

Silver Sensores: Estructuras Aplanadas.


In [0]:
%sql
-- 2. Ver Drivers Sucio
SELECT *
FROM ecomotive.bronze.bronze_drivers 
LIMIT 25;

driver_id,raw_name,license_ref,hourly_wage_text,certifications_list,hired_date,email,_rescued_data
D-101,Miguel Moreno,LIC-816063,22.12 EUR,ElectricVehicle-Hazmat-HeavyLoad,2023-11-25,miguel.moreno@ecomotive.com,null
D-102,CARMEN DIAZ,LIC-697157,37.78,NightShift-ElectricVehicle,2023-06-15,carmen.diaz@ecomotive.com,null
D-103,Elena Navarro,null,23.3,Hazmat-International,2023-11-17,elena.navarro@ecomotive.com,null
D-104,Jorge Alvarez,LIC-273587,34.72 EUR,ElectricVehicle,2023-09-20,jorge.alvarez@ecomotive.com,null
D-105,Elena Romero,LIC-319656,22.0 EUR,International-HeavyLoad-NightShift,2023-09-24,elena.romero@ecomotive.com,null
D-106,Laura Lopez,LIC-768741,47.51,HeavyLoad-Hazmat-NightShift,2023-06-21,laura.lopez@ecomotive.com,null
D-107,isabel ruiz,LIC-335945,30.62,International-Hazmat-HeavyLoad,2023-09-04,isabel.ruiz@ecomotive.com,null
D-108,David Alvarez,LIC-140454,31.12,International-HeavyLoad-ElectricVehicle,2023-04-17,david.alvarez@ecomotive.com,null
D-109,CARLOS HERNANDEZ,LIC-850449,28.91,HeavyLoad,2023-07-26,carlos.hernandez@ecomotive.com,null
D-110,ana diaz,LIC-468978,32.16 EUR,ElectricVehicle,2023-01-15,ana.diaz@ecomotive.com,null


In [0]:
%sql
DESCRIBE HISTORY ecomotive.silver.silver_drivers 

In [0]:
%sql
-- 2. Ver Drivers Limpios
SELECT *
FROM ecomotive.silver.silver_drivers 
LIMIT 25;


driver_id,driver_name,license_ref,hourly_wage,certifications,hire_date,processed_at
D-101,Miguel Moreno,LIC-816063,22.12,"List(ElectricVehicle, Hazmat, HeavyLoad)",2023-11-25,2025-12-19T13:32:05.838Z
D-102,Carmen Diaz,LIC-697157,37.78,"List(NightShift, ElectricVehicle)",2023-06-15,2025-12-19T13:32:05.838Z
D-103,Elena Navarro,null,23.3,"List(Hazmat, International)",2023-11-17,2025-12-19T13:32:05.838Z
D-104,Jorge Alvarez,LIC-273587,34.72,List(ElectricVehicle),2023-09-20,2025-12-19T13:32:05.838Z
D-105,Elena Romero,LIC-319656,22.0,"List(International, HeavyLoad, NightShift)",2023-09-24,2025-12-19T13:32:05.838Z
D-106,Laura Lopez,LIC-768741,47.51,"List(HeavyLoad, Hazmat, NightShift)",2023-06-21,2025-12-19T13:32:05.838Z
D-107,Isabel Ruiz,LIC-335945,30.62,"List(International, Hazmat, HeavyLoad)",2023-09-04,2025-12-19T13:32:05.838Z
D-108,David Alvarez,LIC-140454,31.12,"List(International, HeavyLoad, ElectricVehicle)",2023-04-17,2025-12-19T13:32:05.838Z
D-109,Carlos Hernandez,LIC-850449,28.91,List(HeavyLoad),2023-07-26,2025-12-19T13:32:05.838Z
D-110,Ana Diaz,LIC-468978,32.16,List(ElectricVehicle),2023-01-15,2025-12-19T13:32:05.838Z


In [0]:
%sql
-- 3. Ver Sensores BRONZE
SELECT *
FROM ecomotive.bronze.bronze_sensors 
LIMIT 25;

battery_level,cargo_weight_kg,driver_id,engine_data,error_codes,event_id,location,sensor_id,timestamp,vibration_level,_rescued_data
75.69,2682,D-245,"List(2336, WARNING, 96.0)",List(),evt_200000,"List(36.8566, -7.3768)",TRUCK-013,2025-10-01T13:00:00,0.873,null
67.8,6122,D-112,"List(2459, OK, 80.2)",List(),evt_200001,"List(37.3111, -7.3216)",TRUCK-004,2025-10-01T13:02:00,0.395,null
82.92,4754,D-189,"List(1773, OK, 90.5)",List(),evt_200002,"List(40.0859, -3.4985)",TRUCK-012,2025-10-01T13:04:00,0.144,null
78.94,2808,D-298,"List(2772, OK, 89.7)",List(),evt_200003,"List(39.3351, 0.5495)",TRUCK-001,2025-10-01T13:06:00,0.739,null
86.74,6936,D-156,"List(2772, OK, 88.4)",List(),evt_200004,"List(40.7295, 1.5141)",TRUCK-003,2025-10-01T13:08:00,0.355,null
91.87,7063,D-210,"List(2614, WARNING, 83.0)",List(),evt_200005,"List(36.8781, 2.7364)",TRUCK-020,2025-10-01T13:10:00,0.724,null
31.63,3249,D-134,"List(1585, OK, 93.6)","List(203, 500)",evt_200006,"List(36.4348, -8.9718)",TRUCK-014,2025-10-01T13:12:00,0.011,null
41.31,1306,D-156,"List(1845, OK, 85.6)",List(),evt_200007,"List(41.8181, -0.7336)",TRUCK-003,2025-10-01T13:14:00,0.066,null
29.82,4340,D-277,"List(1698, OK, 82.9)",List(),evt_200008,"List(36.5717, 2.2565)",TRUCK-008,2025-10-01T13:16:00,0.385,null
41.7,4245,D-210,"List(1266, OK, 91.0)","List(404, 101)",evt_200009,"List(39.1011, -4.1225)",TRUCK-020,2025-10-01T13:18:00,0.365,null


In [0]:
%sql
-- 4. Ver Sensores Aplanados
SELECT *
FROM ecomotive.silver.silver_sensors 
LIMIT 25;

event_id,sensor_id,driver_id,event_time,rpm,temperature,engine_status,battery_level,latitude,longitude,error_codes
evt_200000,TRUCK-013,D-245,2025-10-01T13:00:00.000Z,2336,96.0,WARNING,75.69,36.8566,-7.3768,List()
evt_200001,TRUCK-004,D-112,2025-10-01T13:02:00.000Z,2459,80.2,OK,67.8,37.3111,-7.3216,List()
evt_200002,TRUCK-012,D-189,2025-10-01T13:04:00.000Z,1773,90.5,OK,82.92,40.0859,-3.4985,List()
evt_200003,TRUCK-001,D-298,2025-10-01T13:06:00.000Z,2772,89.7,OK,78.94,39.3351,0.5495,List()
evt_200004,TRUCK-003,D-156,2025-10-01T13:08:00.000Z,2772,88.4,OK,86.74,40.7295,1.5141,List()
evt_200005,TRUCK-020,D-210,2025-10-01T13:10:00.000Z,2614,83.0,WARNING,91.87,36.8781,2.7364,List()
evt_200006,TRUCK-014,D-134,2025-10-01T13:12:00.000Z,1585,93.6,OK,31.63,36.4348,-8.9718,"List(203, 500)"
evt_200007,TRUCK-003,D-156,2025-10-01T13:14:00.000Z,1845,85.6,OK,41.31,41.8181,-0.7336,List()
evt_200008,TRUCK-008,D-277,2025-10-01T13:16:00.000Z,1698,82.9,OK,29.82,36.5717,2.2565,List()
evt_200009,TRUCK-020,D-210,2025-10-01T13:18:00.000Z,1266,91.0,OK,41.7,39.1011,-4.1225,"List(404, 101)"
